## Task 1: Text-to-Math Agent Overview

1. A Text-to-Math problem is a maths question written in normal sentences. We need to understand the question and use maths to find the answer.


2. Agents help us solve maths problems step by step. They understand the question, decide which calculation is needed, use a calculator if required, and give the final answer.


3. A normal LLM tries to answer a question directly. Sometimes it may make mistakes in calculations. An agent can use tools like a calculator to perform calculations more accurately. It can also break a problem into smaller steps before giving the answer.

## Task 2: Build Text-to-Math Agent

In [2]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import re

load_dotenv()

model = ChatGroq(model='openai/gpt-oss-120b')

# Using latest approach as older approach is not available
@tool
def calculator(expression: str) -> str:
    """Use this tool for mathematical questions."""
    math_expr = ''.join(re.findall(r'[\d\.\+\-\*\/\(\)\s]+', expression))
    return str(eval(math_expr, {"__builtins__": {}}, {}))

sys_prompt = """
You are a Text-to-Math Agent.

Follow these steps when solving a problem:

1. Understand the word problem.
2. Identify the given numbers and what needs to be calculated.
3. Break the problem into clear steps.
4. Use the calculator tool for arithmetic calculations.
5. Explain the calculation and provide the final answer.

For algebra problems, explain the equation and solve it
step by step. Use the calculator for numerical calculations.
"""

agent = create_agent(model=model, tools=[calculator], system_prompt=sys_prompt)

In [ ]:
## Test Percentage problem
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "A mobile phone costs ₹15,000. The shop offers "
                "a 20% discount. Calculate the discount and "
                "the final price."
            )
        }
    ]
})

print(response["messages"][-1].content)

**Step 1: Identify the given information**  
- Original price of the phone = ₹15,000  
- Discount offered = 20 %  

**Step 2: Calculate the discount amount**  
Discount = 20 % of ₹15,000  
\[
\text{Discount}=15000 \times 0.20 = ₹3,000
\]

**Step 3: Find the final price after discount**  
Final price = Original price – Discount  
\[
\text{Final price}=15000 - 3000 = ₹12,000
\]

**Answer**  
- Discount amount: **₹3,000**  
- Final price of the mobile phone: **₹12,000**.


In [ ]:
# Test Arithmetic problem

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "A shopkeeper has 50 apples. He sells 18 apples "
                "in the morning and 12 in the evening. "
                "How many apples are left?"
            )
        }
    ]
})

print(response["messages"][-1].content)

**Step 1: Understand the problem**  
The shopkeeper starts with 50 apples. He sells some in the morning and some in the evening. We need to find how many apples remain.

**Step 2: Identify the numbers**  
- Initial apples = 50  
- Apples sold in the morning = 18  
- Apples sold in the evening = 12  

**Step 3: Set up the calculation**  
Total apples sold = 18 + 12 = 30.  
Apples left = Initial apples – Total apples sold = 50 − 30.

**Step 4: Perform the arithmetic** (using the calculator)  

\(50 - (18 + 12) = 20\).

**Answer:** The shopkeeper has **20 apples left**.


In [6]:
# Test Simple algebra problem

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "Solve the equation 3x + 5 = 20. "
                "Explain the steps."
            )
        }
    ]
})

print(response["messages"][-1].content)

**Step 1: Write down the equation**  
\(3x + 5 = 20\)

**Step 2: Isolate the term with \(x\)**  
Subtract 5 from both sides to move the constant term to the right side.

\[
3x + 5 - 5 = 20 - 5 \quad\Longrightarrow\quad 3x = 15
\]

(The calculator confirmed that \(20-5 = 15\).)

**Step 3: Solve for \(x\)**  
Divide both sides by the coefficient of \(x\), which is 3.

\[
\frac{3x}{3} = \frac{15}{3} \quad\Longrightarrow\quad x = 5
\]

**Answer:**  
\[
\boxed{x = 5}
\]


## Task 3: Session State for Application

Please refer app.py